In [1]:
from pyspark.sql import SparkSession

PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Permormance_Analytics") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [ ]:
from pyspark.sql.functions import *

spark.sql("""
SELECT 
    COUNT(*) as total_files,
    ROUND(AVG(file_size_in_bytes) / 1024, 2) as avg_size_kb,
    SUM(record_count) as total_records
FROM my_catalog.default.user_events_time_based_partitioned.files
""").show()

spark.sql("""
SELECT 
    file_path, 
    file_size_in_bytes, 
    record_count 
FROM my_catalog.default.user_events_time_based_partitioned.files 
LIMIT 5
""").show(truncate=False)

In [20]:
import time
import os 
from datetime import datetime

LOG_FILE = "benchmark_results_time_based_partitioned.csv"

if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w") as f:
        f.write("timestamp,files_count,execution_time_agg,execution_time_single\n")
    print(f"log file created: {LOG_FILE}")

def benchmark_read(table_name="my_catalog.default.user_events_time_based_partitioned"):
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"run ts: {current_time}")

    
    start_time_agg = time.time()
    spark.sql(f"""
        SELECT 
            shop,
            device_type,
            event_type,
            COUNT(DISTINCT session_id) as unique_sessions, 
            COUNT(DISTINCT user_id) as unique_users,
            SUM(price) as total_revenue, 
            AVG(price) as avg_check,
            MIN(timestamp) as first_interaction,
            MAX(timestamp) as latest_interaction
        FROM {table_name}
        WHERE shop IN ('rozetka', 'foxtrot', 'citrus') AND device_type IN ('ios', 'android')
        GROUP BY 1,2,3
        ORDER BY 
            total_revenue DESC,
            unique_sessions DESC
    """).collect() # action for spark to actually calculate things
    
    end_time_agg = time.time()
    execution_time_agg = end_time_agg - start_time_agg
    files_count = spark.sql("SELECT COUNT(*) FROM my_catalog.default.user_events_time_based_partitioned.files").collect()[0][0]

    start_time_single = time.time()
    spark.sql(f"""
        SELECT * FROM {table_name}
        WHERE shop = 'rozetka' 
          AND device_type = 'mobile_web' 
          AND event_type = 'purchase'
    """).collect()
    end_time_single = time.time()

    execution_time_single = end_time_single - start_time_single

    with open(LOG_FILE, "a") as f:
        f.write(f"{current_time},{files_count},{execution_time_agg:.4f},{execution_time_single:.4f}\n")
    
    print(f"FILES NUMBER: {files_count} /// TIME EXECUTING (AGG): {execution_time_agg:.4f}")
    print(f"FILES NUMBER: {files_count} /// TIME EXECUTING (SINGLE): {execution_time_single:.4f}")


In [26]:

INTERVAL_MINUTES = 15

try:
    while True:
        benchmark_read()
        print(f"waiting set interval")
        time.sleep(INTERVAL_MINUTES * 60)
except KeyboardInterrupt:
    print("interrupted")

run ts: 2026-05-22 18:16:44
FILES NUMBER: 2 /// TIME EXECUTING (AGG): 0.6021
FILES NUMBER: 2 /// TIME EXECUTING (SINGLE): 0.1228
waiting set interval
run ts: 2026-05-22 18:31:45
FILES NUMBER: 185 /// TIME EXECUTING (AGG): 2.9778
FILES NUMBER: 185 /// TIME EXECUTING (SINGLE): 1.4559
waiting set interval
run ts: 2026-05-22 18:46:50
FILES NUMBER: 65 /// TIME EXECUTING (AGG): 2.2739
FILES NUMBER: 65 /// TIME EXECUTING (SINGLE): 0.5516
waiting set interval
run ts: 2026-05-22 19:01:53
FILES NUMBER: 246 /// TIME EXECUTING (AGG): 4.5369
FILES NUMBER: 246 /// TIME EXECUTING (SINGLE): 1.8313
waiting set interval
run ts: 2026-05-22 19:17:00
FILES NUMBER: 427 /// TIME EXECUTING (AGG): 8.3842
FILES NUMBER: 427 /// TIME EXECUTING (SINGLE): 3.3015
waiting set interval
run ts: 2026-05-22 19:32:12
FILES NUMBER: 245 /// TIME EXECUTING (AGG): 4.5131
FILES NUMBER: 245 /// TIME EXECUTING (SINGLE): 2.0464
waiting set interval
run ts: 2026-05-22 19:47:19
FILES NUMBER: 406 /// TIME EXECUTING (AGG): 5.8212
FIL

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


run ts: 2026-05-23 03:15:53


Py4JError: PythonUtils does not exist in the JVM

In [ ]:
run ts: 2026-05-22 18:16:44
FILES NUMBER: 2 /// TIME EXECUTING (AGG): 0.6021
FILES NUMBER: 2 /// TIME EXECUTING (SINGLE): 0.1228
waiting set interval
run ts: 2026-05-22 18:31:45
FILES NUMBER: 185 /// TIME EXECUTING (AGG): 2.9778
FILES NUMBER: 185 /// TIME EXECUTING (SINGLE): 1.4559
waiting set interval
run ts: 2026-05-22 18:46:50
FILES NUMBER: 65 /// TIME EXECUTING (AGG): 2.2739
FILES NUMBER: 65 /// TIME EXECUTING (SINGLE): 0.5516
waiting set interval
run ts: 2026-05-22 19:01:53
FILES NUMBER: 246 /// TIME EXECUTING (AGG): 4.5369
FILES NUMBER: 246 /// TIME EXECUTING (SINGLE): 1.8313
waiting set interval
run ts: 2026-05-22 19:17:00
FILES NUMBER: 427 /// TIME EXECUTING (AGG): 8.3842
FILES NUMBER: 427 /// TIME EXECUTING (SINGLE): 3.3015
waiting set interval
run ts: 2026-05-22 19:32:12
FILES NUMBER: 245 /// TIME EXECUTING (AGG): 4.5131
FILES NUMBER: 245 /// TIME EXECUTING (SINGLE): 2.0464
waiting set interval
run ts: 2026-05-22 19:47:19
FILES NUMBER: 406 /// TIME EXECUTING (AGG): 5.8212
FILES NUMBER: 406 /// TIME EXECUTING (SINGLE): 4.3145
waiting set interval
run ts: 2026-05-22 20:02:30
FILES NUMBER: 225 /// TIME EXECUTING (AGG): 4.6086
FILES NUMBER: 225 /// TIME EXECUTING (SINGLE): 1.6122
waiting set interval
run ts: 2026-05-22 20:17:37
FILES NUMBER: 406 /// TIME EXECUTING (AGG): 6.5219
FILES NUMBER: 406 /// TIME EXECUTING (SINGLE): 2.7883
waiting set interval
run ts: 2026-05-22 20:32:47
FILES NUMBER: 241 /// TIME EXECUTING (AGG): 5.3585
FILES NUMBER: 241 /// TIME EXECUTING (SINGLE): 2.0085
waiting set interval
run ts: 2026-05-22 20:47:55
FILES NUMBER: 422 /// TIME EXECUTING (AGG): 7.3905
FILES NUMBER: 422 /// TIME EXECUTING (SINGLE): 2.9660
waiting set interval
run ts: 2026-05-22 21:03:06
FILES NUMBER: 241 /// TIME EXECUTING (AGG): 5.3838
FILES NUMBER: 241 /// TIME EXECUTING (SINGLE): 1.7716
waiting set interval
run ts: 2026-05-22 21:18:13
FILES NUMBER: 422 /// TIME EXECUTING (AGG): 6.9939
FILES NUMBER: 422 /// TIME EXECUTING (SINGLE): 3.0224
waiting set interval
run ts: 2026-05-22 21:33:24
FILES NUMBER: 238 /// TIME EXECUTING (AGG): 6.1279
FILES NUMBER: 238 /// TIME EXECUTING (SINGLE): 1.9397
waiting set interval
run ts: 2026-05-22 21:48:32
FILES NUMBER: 420 /// TIME EXECUTING (AGG): 7.9787
FILES NUMBER: 420 /// TIME EXECUTING (SINGLE): 2.9372
waiting set interval
run ts: 2026-05-22 22:03:44
FILES NUMBER: 239 /// TIME EXECUTING (AGG): 5.9224
FILES NUMBER: 239 /// TIME EXECUTING (SINGLE): 2.1877
waiting set interval
run ts: 2026-05-22 22:18:52
FILES NUMBER: 421 /// TIME EXECUTING (AGG): 7.2338
FILES NUMBER: 421 /// TIME EXECUTING (SINGLE): 3.3399
waiting set interval
run ts: 2026-05-22 22:34:03
FILES NUMBER: 238 /// TIME EXECUTING (AGG): 7.0131
FILES NUMBER: 238 /// TIME EXECUTING (SINGLE): 2.3157
waiting set interval
run ts: 2026-05-22 22:49:13
FILES NUMBER: 420 /// TIME EXECUTING (AGG): 8.9596
FILES NUMBER: 420 /// TIME EXECUTING (SINGLE): 3.5735
waiting set interval
run ts: 2026-05-22 23:04:26
FILES NUMBER: 239 /// TIME EXECUTING (AGG): 6.7672
FILES NUMBER: 239 /// TIME EXECUTING (SINGLE): 2.8178
waiting set interval
run ts: 2026-05-22 23:19:36
FILES NUMBER: 38 /// TIME EXECUTING (AGG): 1.0151
FILES NUMBER: 38 /// TIME EXECUTING (SINGLE): 0.4827
waiting set interval
run ts: 2026-05-22 23:34:38
FILES NUMBER: 218 /// TIME EXECUTING (AGG): 3.3198
FILES NUMBER: 218 /// TIME EXECUTING (SINGLE): 1.2342
waiting set interval
run ts: 2026-05-22 23:49:43
FILES NUMBER: 399 /// TIME EXECUTING (AGG): 5.2545
FILES NUMBER: 399 /// TIME EXECUTING (SINGLE): 2.5101
waiting set interval
run ts: 2026-05-23 00:04:51
FILES NUMBER: 582 /// TIME EXECUTING (AGG): 6.8877
FILES NUMBER: 582 /// TIME EXECUTING (SINGLE): 3.2975
waiting set interval
run ts: 2026-05-23 00:20:02
FILES NUMBER: 764 /// TIME EXECUTING (AGG): 8.4748
FILES NUMBER: 764 /// TIME EXECUTING (SINGLE): 4.2773
waiting set interval
run ts: 2026-05-23 00:35:16
FILES NUMBER: 947 /// TIME EXECUTING (AGG): 10.1621
FILES NUMBER: 947 /// TIME EXECUTING (SINGLE): 5.1000
waiting set interval
run ts: 2026-05-23 00:50:33
FILES NUMBER: 1130 /// TIME EXECUTING (AGG): 12.5121
FILES NUMBER: 1130 /// TIME EXECUTING (SINGLE): 6.1715
waiting set interval
run ts: 2026-05-23 01:05:54
FILES NUMBER: 1314 /// TIME EXECUTING (AGG): 13.5878
FILES NUMBER: 1314 /// TIME EXECUTING (SINGLE): 7.2834
waiting set interval
run ts: 2026-05-23 01:21:16
FILES NUMBER: 105 /// TIME EXECUTING (AGG): 4.2420
FILES NUMBER: 105 /// TIME EXECUTING (SINGLE): 1.3724
waiting set interval
run ts: 2026-05-23 01:36:22
FILES NUMBER: 286 /// TIME EXECUTING (AGG): 5.3343
FILES NUMBER: 286 /// TIME EXECUTING (SINGLE): 2.0033
waiting set interval
run ts: 2026-05-23 01:51:30
FILES NUMBER: 468 /// TIME EXECUTING (AGG): 8.1208
FILES NUMBER: 468 /// TIME EXECUTING (SINGLE): 3.8912
waiting set interval
run ts: 2026-05-23 02:06:43
FILES NUMBER: 651 /// TIME EXECUTING (AGG): 8.8321
FILES NUMBER: 651 /// TIME EXECUTING (SINGLE): 4.1873
waiting set interval
run ts: 2026-05-23 02:21:57
FILES NUMBER: 834 /// TIME EXECUTING (AGG): 11.6357
FILES NUMBER: 834 /// TIME EXECUTING (SINGLE): 5.1230
waiting set interval
run ts: 2026-05-23 02:37:15
FILES NUMBER: 1017 /// TIME EXECUTING (AGG): 12.9731
FILES NUMBER: 1017 /// TIME EXECUTING (SINGLE): 6.3733
waiting set interval
run ts: 2026-05-23 02:52:36
FILES NUMBER: 1202 /// TIME EXECUTING (AGG): 15.0433
FILES NUMBER: 1202 /// TIME EXECUTING (SINGLE): 8.6136
waiting set interval